# Part teòrica

**Variables:** Caselles consecutives amb direcció vertical o horitzontal de longitud > 1

**Domini:** Paraules del diccionari

**Restriccions:**
- Files i columnes han d'estar dins del taulell
- Files i columnes > 1 casella
- Interseccions de files i columnes tenen la mateixa lletra
- Si es troba una # acaba la fila o la columna
- Les paraules han d'estar escrites de dalt a baix i d'esquerra a dreta
- No es pot repetir una paraula
- Les paraules han de tenir una llargada menor o igual al màxim de m o n del taulell

**Tamany espai de solucions inicial:** D^v on *D* es el domini de paraules del diccionari i *v* les variables

**Estratègia de millora:** 


# Codi

## Comú

### Carregar biblioteques

In [62]:
import numpy as np
import timeit
import copy

### Loading data

In [63]:
def loadCrossword(file):
    data = []
    with open(file, 'r') as file:
        for line in file.readlines():
            elements = line.strip().split()
            data.append(elements)
    return(np.array(data))

def loadDictionary(file):
    words = {}
    with open(file, 'r') as file:
        for word in file.readlines():   
            if len(word.strip()) not in words.keys():
                words[len(word.strip())] = [word.strip()]
            else:
                v = words[len(word.strip())]
                v.append(word.strip())

    return words

In [64]:
def cercaVariablesHoritzontal(taulell, variables):
    for n, i in enumerate(taulell):
        length = 0
        initial_pos = [n, 0]
        for m, j in enumerate(i):
            if j == '#':
                if length > 1:
                    variables.append([initial_pos, length, 0])
                length = 0
                initial_pos = [n, m + 1]
            elif m == len(i) - 1:
                length += 1
                if length > 1:
                    variables.append([initial_pos, length, 0])
                length = 0
                initial_pos = [n, m + 1]
            else: length += 1

### Cerca de variables
```variable = [[pos_inicial], len, v/h]; vertical = 1, horitzontal = 0 ```

In [65]:
def cercaVariablesVertical(taulell, variables):
    transposed_taulell = taulell.transpose()
    for n, i in enumerate(transposed_taulell):
        length = 0
        initial_pos = [n, 0]
        for m, j in enumerate(i):
            if j == '#':
                if length > 1:
                    initial_pos.reverse()
                    variables.append([initial_pos, length, 1])
                length = 0
                initial_pos = [n, m + 1]
            elif m == len(i) - 1:
                length += 1
                if length > 1:
                    initial_pos.reverse()
                    variables.append([initial_pos, length, 1])
                length = 0
                initial_pos = [n, m + 1]
            else: length += 1

In [66]:
def cercaVariables(taulell, variables):
    cercaVariablesHoritzontal(taulell, variables)
    cercaVariablesVertical(taulell, variables)

In [67]:
def calculaPosicionsVariable(variable):
    variable_positions = []
    for x in range(variable[1]):
        i, j = 0, 0
        if variable[2] == 0: i = x
        else: j = x
        variable_positions.append([variable[0][0]+j, variable[0][1]+i])
    
    return variable_positions

### Letter of intersection

In [68]:
def intersectLetter(variable, word, assigned_variable, pos):
    if variable[2] == 0:
        return assigned_variable[1][pos[0] - assigned_variable[0][0][0]] == word[pos[1] - variable[0][1]]
    elif variable[2] == 1:
        return assigned_variable[1][pos[1] - assigned_variable[0][0][1]] == word[pos[0] - variable[0][0]]
    else: return False

### Comprovació de restriccions

In [69]:
def isValid(word, variable, assigned_variables):

    for assigned_variable in assigned_variables:    
        if word == assigned_variable[1]:
            return False
        
    for var in assigned_variables:
        for intersection in variable[3]:
            if intersection in var[0][3] and not intersectLetter(variable, word, var, intersection): return False
    return True

## Exercici 1

### Backtracking

```
Funcio Backtracking(LVA,LVNA,R,D)
    Si (LVNA és buida) llavors Retornar(LVA) fSi
    Var=Cap(LVNA);
    Per a cada (valor del Domini(Var, D) que podem assignar a Var) fer
        Si (SatisfaRestriccions([Var valor],LVA,R)) llavors
            Res=Backtracking(Insertar([Var, valor],LVA),Cua(LVNA),R,D);
            Si (Res és una solució completa) llavors
                Retornar(Res);
            Fsi
        Fsi
    Fper
    Retornar(Falla)
FFuncio

```

In [70]:
def backtracking(assigned_variables, variables, dictionary):
    if not variables:
        return assigned_variables

    var = variables[0]
    for word in dictionary[var[1]]:
        if isValid(word, var, assigned_variables):
            assigned_variables.append([var, word])
            result = backtracking(assigned_variables, variables[1:], dictionary)

            if result:
                return result
 
            assigned_variables.pop()  

    return None

## Exercici 2

### Inicialitzar dominis FC

In [71]:
def initDomains(variables, dictionary):
    fcDomains = {}
    for i, var in enumerate(variables):
        fcDomains[i] = list(dictionary[var[1]])
    return fcDomains

### ActualitzarDominis

In [72]:
def actualitzarDominis(variables, var, word, fcDomains, index, assigned_variables, originalDomains):
    for i in range(1, len(variables)):
        if variables[i] not in [assigned[0] for assigned in assigned_variables]:
            for position in variables[i][3]:
                if position in var[3]:
                    auxDomain = fcDomains[i + index][:]
                    for possible_word in fcDomains[i + index]:
                        if possible_word == word or not intersectLetter(variables[i], possible_word, [var, word], position):
                            auxDomain.remove(possible_word)
                    if not auxDomain:
                        fcDomains[i + index] = originalDomains[i + index][:]
                        return None
                    
                    fcDomains[i + index] = auxDomain[:]

    return True

### CalculateMRV

In [ ]:
def calculateMRV(variables, assigned_variables, fcDomains):
    unassigned_variables = [var for var in variables if var not in [assigned[0] for assigned in assigned_variables]]
    mrv_values = [(var, len(fcDomains[i])) for i, var in enumerate(unassigned_variables)]
    mrv_values.sort(key=lambda x: x[1])
    return [var[0] for var in mrv_values]

### ForwardChecking

In [73]:
def forwardChecking(assigned_variables, variables, fcDomains, originalDomains):
    if not variables:
        return assigned_variables
    
    var = variables[0]
    for word in fcDomains[len(assigned_variables)]:
        if isValid(word, var, assigned_variables) and actualitzarDominis(variables, var, word, fcDomains, len(assigned_variables), assigned_variables, originalDomains):
            assigned_variables.append([var, word])
            result = forwardChecking(assigned_variables, variables[1:], fcDomains, originalDomains)

            if result:
                return result

            assigned_variables.pop()

    return None

### Trobar interseccions

In [74]:
def trobaInterseccions(variables):
    posicions = []
    interseccions = []
    
    for variable in variables:
        for pos in calculaPosicionsVariable(variable):
            if pos in posicions: interseccions.append(pos)
            else: posicions.append(pos)
            
    for variable in variables:
        aux_interseccions = []
        for pos in calculaPosicionsVariable(variable):
            if pos in interseccions: aux_interseccions.append(pos)
        variable.append(aux_interseccions)

### Print taulell final

In [75]:
def printSolution(assigned_variables, taulell):
    for variable, word in assigned_variables:
        positions = calculaPosicionsVariable(variable)
        for i, j in positions:
            taulell[i][j] = word[i - variable[0][0] if variable[2] == 1 else j - variable[0][1]]

    for row in taulell:
        print(" ".join(row))

### Main

In [76]:
if __name__ == '__main__':
    #Configuració general
    taulell = loadCrossword('C:\\Users\\ppugi\\Desktop\\Desktop\\UNI\\CRI\\PractiquesCRI\\Practica1\\crossword_CB_v3.txt')
    dictionary = loadDictionary('C:\\Users\\ppugi\\Desktop\\Desktop\\UNI\\CRI\\PractiquesCRI\\Practica1\\diccionari_CB_v3.txt')
    
    #Backtracking----------------------------------------------------------------------------------------------------------------------
    assigned_variables = []
    variables = []
    cercaVariables(taulell, variables)
    trobaInterseccions(variables)
    
    bc = backtracking(assigned_variables, variables, dictionary)

    print("Resultat backtracking: ", bc)
    
    printSolution(assigned_variables, taulell)
    
    #Forward Checking-------------------------------------------------------------------------------------------------------------------
    assigned_variables = []
    variables = []
    cercaVariables(taulell, variables)
    trobaInterseccions(variables)
    fcDomains = initDomains(variables, dictionary)
    originalDomains = initDomains(variables, dictionary)
    
    FC = forwardChecking(assigned_variables, variables, fcDomains, originalDomains)
    
    print("\n\nResultat FC:", FC)
    printSolution(assigned_variables, taulell)

Resultat backtracking:  [[[[0, 0], 6, 0, [[0, 0], [0, 3], [0, 5]]], 'CANTAR'], [[[2, 2], 4, 0, [[2, 3], [2, 5]]], 'CLAN'], [[[4, 1], 5, 0, [[4, 1], [4, 2], [4, 3], [4, 5]]], 'PREMI'], [[[5, 0], 4, 0, [[5, 0], [5, 1], [5, 2], [5, 3]]], 'PIAR'], [[[6, 0], 2, 0, [[6, 0], [6, 1]]], 'ON'], [[[0, 0], 4, 1, [[0, 0]]], 'CARA'], [[[5, 0], 2, 1, [[5, 0], [6, 0]]], 'PO'], [[[4, 1], 3, 1, [[4, 1], [5, 1], [6, 1]]], 'PIN'], [[[4, 2], 2, 1, [[4, 2], [5, 2]]], 'RA'], [[[0, 3], 6, 1, [[0, 3], [2, 3], [4, 3], [5, 3]]], 'TALLER'], [[[0, 5], 5, 1, [[0, 5], [2, 5], [4, 5]]], 'RANCI']]
C A N T A R
A # # A # A
R # C L A N
A # # L # C
# P R E M I
P I A R # #
O N # # # #


Resultat FC: [[[[0, 0], 6, 0, [[0, 0], [0, 3], [0, 5]]], 'CANTAR'], [[[2, 2], 4, 0, [[2, 3], [2, 5]]], 'CLAN'], [[[4, 1], 5, 0, [[4, 1], [4, 2], [4, 3], [4, 5]]], 'PREMI'], [[[5, 0], 4, 0, [[5, 0], [5, 1], [5, 2], [5, 3]]], 'PIAR'], [[[6, 0], 2, 0, [[6, 0], [6, 1]]], 'ON'], [[[0, 0], 4, 1, [[0, 0]]], 'CARA'], [[[5, 0], 2, 1, [[5, 0], [6, 0]